In [1]:
from pathlib import Path
import gzip
import shutil
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
file_path = PROJECT_ROOT / "data/raw/emdat/emdat_raw.xlsx"

df = pd.read_excel(file_path)
df.head()

In [ ]:
df.describe()

,AID Contribution ('000 US$),Magnitude,Latitude,Longitude,Start Year,Start Month,Start Day,End Year,End Month,End Day,...,No. Affected,No. Homeless,Total Affected,Reconstruction Costs ('000 US$),"Reconstruction Costs, Adjusted ('000 US$)",Insured Damage ('000 US$),"Insured Damage, Adjusted ('000 US$)",Total Damage ('000 US$),"Total Damage, Adjusted ('000 US$)",CPI
count,2.820000e+02,2.411000e+03,1643.000000,1643.000000,8111.000000,8044.000000,7089.000000,8111.000000,8028.000000,7146.000000,...,5.425000e+03,8.180000e+02,6.554000e+03,3.000000e+01,3.000000e+01,5.700000e+02,5.700000e+02,2.540000e+03,2.540000e+03,8111.000000
mean,4.264987e+04,5.532913e+04,16.055224,40.851893,2014.507089,6.398931,15.058118,2014.559364,6.643000,15.980269,...,6.191307e+05,3.747838e+04,5.179880e+05,6.171692e+06,7.283311e+06,1.567621e+06,2.087260e+06,1.329598e+06,1.749196e+06,75.424099
std,2.746491e+05,3.132158e+05,22.107346,75.650014,5.996375,3.376916,9.059093,5.993802,3.343073,8.949045,...,5.863693e+06,2.596501e+05,5.341139e+06,1.825595e+07,1.943857e+07,5.151087e+06,6.952775e+06,7.020853e+06,9.674779e+06,10.454612
min,9.000000e+00,-5.500000e+01,-72.640000,-172.095000,2005.000000,1.000000,1.000000,2005.000000,1.000000,1.000000,...,2.000000e+00,3.000000e+00,1.000000e+00,8.400000e+01,1.380000e+02,3.400000e+01,5.000000e+01,2.000000e+00,3.000000e+00,60.660332
25%,1.757500e+02,1.800000e+01,0.790000,-1.020000,2009.000000,4.000000,7.000000,2009.000000,4.000000,8.000000,...,1.550000e+03,3.787500e+02,8.600000e+02,1.414110e+05,2.164718e+05,1.000000e+05,1.355678e+05,2.500000e+04,3.196350e+04,66.875974
50%,1.007500e+03,1.940000e+02,18.443000,49.130000,2015.000000,7.000000,15.000000,2015.000000,7.000000,16.000000,...,1.120000e+04,1.983500e+03,7.020500e+03,6.227000e+05,7.732755e+05,3.205000e+05,4.350670e+05,1.400000e+05,1.860380e+05,73.620803
75%,6.372000e+03,2.858891e+04,34.523000,103.180000,2020.000000,9.000000,23.000000,2020.000000,9.000000,24.000000,...,9.000000e+04,7.992750e+03,6.000000e+04,4.736000e+06,7.550644e+06,9.537500e+05,1.349115e+06,7.000000e+05,9.421652e+05,80.390383
max,3.518530e+06,1.302587e+07,67.930000,179.650000,2024.000000,12.000000,31.000000,2025.000000,12.000000,31.000000,...,3.300000e+08,5.000000e+06,3.300000e+08,1.000000e+08,1.056584e+08,6.000000e+07,9.891143e+07,2.100000e+08,3.005613e+08,97.436176


In [ ]:
print(df.info())
print(f'Total record: {len(df)}')
print(f'Total columns: {len(df.columns)}')

<class 'pandas.DataFrame'>
RangeIndex: 8111 entries, 0 to 8110
Data columns (total 47 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   DisNo.                                     8111 non-null   str    
 1   Historic                                   8111 non-null   str    
 2   Classification Key                         8111 non-null   str    
 3   Disaster Group                             8111 non-null   str    
 4   Disaster Subgroup                          8111 non-null   str    
 5   Disaster Type                              8111 non-null   str    
 6   Disaster Subtype                           8111 non-null   str    
 7   External IDs                               3820 non-null   str    
 8   Event Name                                 2011 non-null   str    
 9   ISO                                        8111 non-null   str    
 10  Country                            

In [ ]:
# filter các cột cần thiết

KEEP_COLS = [
    "DisNo.", "Disaster Subgroup", "Disaster Type", "Disaster Subtype",
    "ISO", "Country", "Subregion", "Region",
    "Start Year", "Start Month", "End Year", "End Month",
    "Total Deaths", "No. Injured", "No. Affected", "No. Homeless", "Total Affected",
    "Total Damage ('000 US$)", "Total Damage, Adjusted ('000 US$)", "Event Name",
]

df = df[KEEP_COLS]
df["event_id"] = df["DisNo."].str[:9]
print(df.shape)

(8111, 21)


In [ ]:
# transform tên cột

df.columns = (df.columns.str.strip()
                        .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
                        .str.strip("_")
                        .str.lower())


In [ ]:
# rename lại tên cột
RENAME_MAP = {
    "disno": "dis_no",
    "iso": "country_iso3",
    "country": "country_name",
    "region": "continent",
    "start_year": "year",
    "start_month": "month",
    "total_deaths": "deaths",
    "no_injured": "injured",
    "no_affected": "affected",
    "no_homeless": "homeless",
    "total_damage_000_us": "damage_k_usd",
    "total_damage_adjusted_000_us": "damage_adj_k_usd",
}
df = df.rename(columns= RENAME_MAP)
df.columns

Index(['dis_no', 'disaster_subgroup', 'disaster_type', 'disaster_subtype',
       'country_iso3', 'country_name', 'subregion', 'continent', 'year',
       'month', 'end_year', 'end_month', 'deaths', 'injured', 'affected',
       'homeless', 'total_affected', 'damage_k_usd', 'damage_adj_k_usd',
       'event_name', 'event_id'],
      dtype='str')

In [ ]:
# loại bỏ các giá trị không phải là thiên tai 
print(f'Before eliminating non-natural disasters: {len(df)}')
df = df[~df["disaster_subgroup"].isin(["Biological", "Extra-terrestrial"])].copy()
print(f'After filtering: {len(df)}')

Before eliminating non-natural disasters: 8111
After filtering: 7545


In [ ]:
# check duplicate
print(f"Số dòng trùng lặp: {df.duplicated().sum()}")
print(f"dis_no trùng: {df['dis_no'].duplicated().sum()}")

Số dòng trùng lặp: 0
dis_no trùng: 0


In [ ]:
# làm sạch các chuỗi ký tự
str_cols = df.select_dtypes(include= "str").columns
df[str_cols] = df[str_cols].apply(lambda s:s.str.strip())


# chuẩn hoá lại tên của các event
df["event_name"] = (df["event_name"]
                    .str.replace("'", "", regex=False)             # Hurricane 'Irma' -> Hurricane Irma
                    .str.replace(r"\s*\(.*\)$", "", regex=True)    # bỏ tên địa phương trong ngoặc
                    .str.strip())

#check result
print(df['event_name'])

0                                 NaN
1       Gray fire and the Oregon fire
2                                 NaN
3                     Hurricane Beryl
4                     Hurricane Beryl
                    ...              
8106                              NaN
8107                         Currowan
8108          Tropical cyclone Bulbul
8109                              NaN
8110                              NaN
Name: event_name, Length: 7545, dtype: str


In [ ]:
# transform data type 

int_cols = ["year", "month", "end_year", "end_month",
            "deaths", "injured", "affected", "homeless", "total_affected"]


df[int_cols] =df[int_cols].astype('Int64')

In [ ]:
# check validation
num_cols = ["deaths", "injured", "affected", "homeless", "total_affected",
            "damage_k_usd", "damage_adj_k_usd"]

assert (df[num_cols] < 0).sum().sum() == 0, "Có giá trị âm"
assert df["year"].between(2005, 2024).all(), "Có năm ngoài khung"
assert set(df["continent"]) == {"Africa", "Americas", "Asia", "Europe", "Oceania"}

start = df["year"] * 100 + df["month"].fillna(1)
end = df["end_year"] * 100 + df["end_month"].fillna(12)
assert (end >= start).all(), "Có sự kiện kết thúc trước khi bắt đầu"
print("Tất cả kiểm tra đều đạt")

Tất cả kiểm tra đều đạt


In [ ]:
# xử lý missing value
df["has_deaths_data"] = df["deaths"].notna()
df["has_damage_data"] = df["damage_adj_k_usd"].notna()
df["month_known"] = df["month"].notna()

In [ ]:
# chuyển đổi đơn vị tiền tệ

# giá USD tại năm xảy ra
df["damage_usd_raw"] = df["damage_k_usd"] * 1000      

# sau khi lạm phát thì đã được điều chỉnh
df["damage_usd"] = df["damage_adj_k_usd"] * 1000        
df = df.drop(columns=["damage_k_usd", "damage_adj_k_usd"])

In [ ]:
#check 5 giá trị lớn nhất theo deaths
df.nlargest(5, "deaths")[["dis_no", "disaster_type", "country_name", "year", "deaths"]]

,dis_no,disaster_type,country_name,year,deaths
2888,2010-0017-HTI,Earthquake,Haiti,2010,222570
1922,2008-0184-MMR,Storm,Myanmar,2008,138366
7015,2008-0192-CHN,Earthquake,China,2008,87476
2437,2005-0575-PAK,Earthquake,Pakistan,2005,73338
2158,2010-0372-RUS,Extreme temperature,Russian Federation,2010,55736


In [ ]:
# lưu vào file mới 
out_path = PROJECT_ROOT / "data/clean/emdat_clean.csv"
df.to_csv(out_path, index=False, encoding="utf-8")
print(df.shape, "->", out_path)

(7545, 24) -> k:\TTDL\data\interim\emdat_clean.csv
